# 🔗 Pipeline — Quick Notes
---
> **Dataset:** `tips` (seaborn) + `titanic` | **Library:** `sklearn.pipeline`

## 📌 Key Points
- Chains **multiple steps** (preprocessing + model) into one object
- **Prevents data leakage** — fit/transform only on train automatically
- Makes code cleaner and more reproducible
- Works seamlessly with `GridSearchCV` and `cross_val_score`
- Last step must be an **estimator** (model); all previous steps must have `transform()`
- `Pipeline` = linear chain | `ColumnTransformer` = parallel branches

## 🔑 Structure
```python
Pipeline([
    ('step1', Transformer1()),
    ('step2', Transformer2()),
    ('model', Estimator())
])
```

## 📊 Flow Diagram
```
Raw Data
   ↓
[Step 1: Encoding]
   ↓
[Step 2: Scaling]
   ↓
[Step 3: Model]
   ↓
Prediction
```

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

df = sns.load_dataset('tips')
x = df.drop(columns=['total_bill'])
y = df['total_bill']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
print("x_train shape:", x_train.shape)
print(x_train.dtypes)

In [ ]:
# Step 1: Define column groups
num_cols = ['tip', 'size']
cat_cols = ['sex', 'smoker', 'day', 'time']

# Step 2: ColumnTransformer for preprocessing
preprocessor = ColumnTransformer([
    ('num', StandardScaler(),                                  num_cols),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_cols)
])

# Step 3: Full Pipeline = preprocessor + model
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model',        LinearRegression())
])

print("Pipeline steps:")
for name, step in pipeline.steps:
    print(f"  {name}: {step}")

In [ ]:
# Fit pipeline on train ONLY
pipeline.fit(x_train, y_train)

# Predict — pipeline handles all transformations automatically
y_pred = pipeline.predict(x_test)
print("R2 Score:", round(r2_score(y_test, y_pred), 4))

In [ ]:
# Access individual steps
print("Scaler mean (tip, size):", pipeline['preprocessor'].named_transformers_['num'].mean_)
print("OHE categories:", pipeline['preprocessor'].named_transformers_['cat'].categories_)

In [ ]:
# Pipeline with GridSearchCV — hyperparameter tuning
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor

pipeline_rf = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42))
])

# Note: use 'model__param' syntax to tune model inside pipeline
param_grid = {
    'model__n_estimators': [50, 100],
    'model__max_depth':    [3, 5]
}

grid = GridSearchCV(pipeline_rf, param_grid, cv=3, scoring='r2')
grid.fit(x_train, y_train)

print("Best Params :", grid.best_params_)
print("Best R2     :", round(grid.best_score_, 4))
print("Test R2     :", round(r2_score(y_test, grid.predict(x_test)), 4))

In [ ]:
# Titanic — full Pipeline for classification
df_t = pd.read_csv("/content/titanic - titanic.csv")
df_t = df_t.drop(columns=['Cabin','Name','Ticket','PassengerId'], errors='ignore')
df_t['Age'].fillna(df_t['Age'].median(), inplace=True)
df_t['Embarked'].fillna(df_t['Embarked'].mode()[0], inplace=True)
df_t.dropna(inplace=True)

num_t = ['Age', 'Fare', 'SibSp', 'Parch', 'Pclass']
cat_t = ['Sex', 'Embarked']

x_t = df_t[num_t + cat_t]
y_t = df_t['Survived']

x_tr, x_te, y_tr, y_te = train_test_split(x_t, y_t, test_size=0.2, random_state=42)

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

pre_t = ColumnTransformer([
    ('num', StandardScaler(),                                  num_t),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_t)
])

pipe_t = Pipeline([
    ('prep',  pre_t),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42))
])

pipe_t.fit(x_tr, y_tr)
print("Titanic Accuracy:", round(accuracy_score(y_te, pipe_t.predict(x_te)), 4))

In [ ]:
# Save & Load Pipeline (for deployment)
import joblib

joblib.dump(pipeline, '/content/tips_pipeline.pkl')
loaded = joblib.load('/content/tips_pipeline.pkl')
print("Loaded pipeline R2:", round(r2_score(y_test, loaded.predict(x_test)), 4))
print("Pipeline saved and loaded successfully ✅")

## 🔧 Key Methods
| Method | Meaning |
|---|---|
| `pipe.fit(x_train, y_train)` | Fit all steps |
| `pipe.predict(x_test)` | Transform + predict |
| `pipe.fit_transform(x_train)` | Fit + transform (no model) |
| `pipe['step_name']` | Access a specific step |
| `pipe.set_params(model__n_estimators=200)` | Change params |

## 🥇 Remember
- **No data leakage** — pipeline fits each step only on train
- Use `'stepname__param'` syntax for `GridSearchCV`
- `joblib.dump()` → save entire pipeline for deployment
- Last step = estimator; others = transformers
- ✅ Always build pipelines in real projects — not optional!